# 06 — Evaluation

**Sentimentanalys av flygbolagstweets — steg 6/6**

Sista notebooken i pipelinen: vi laddar båda de tränade modellerna (från notebook 04 och 05)
och utvärderar dem ordentligt på testdata de aldrig sett — steg 6 i vårt transfer
learning-flöde. Vi knyter också ihop säcken med resultatet från den oförvanskade klustringen
i notebook 03.

> **Kräver internet + GPU (samma session som tidigare):** modellerna är sparade lokalt, men
> `TFDistilBertModel`-arkitekturen laddas via `transformers`, så kör i **Google Colab**.


## Installation & imports

In [ ]:
!pip install -q transformers tensorflow scikit-learn pandas matplotlib seaborn


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_style("whitegrid")


## Del A — Utvärdera sentimentmodellen (notebook 04)

In [ ]:
test_df = pd.read_csv("data/test.csv")
sentiment_model = tf.keras.models.load_model("models/sentiment_model.keras")

from transformers import DistilBertTokenizerFast
sentiment_tokenizer = DistilBertTokenizerFast.from_pretrained("models/sentiment_tokenizer")

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}
MAX_LEN = 64

def tokenize(texts, tok):
    enc = tok(list(texts), max_length=MAX_LEN, truncation=True, padding="max_length", return_tensors="tf")
    return enc["input_ids"], enc["attention_mask"]

X_test_ids, X_test_mask = tokenize(test_df["text"], sentiment_tokenizer)
y_test = test_df["airline_sentiment"].map(label2id).values

print("Testset:", test_df.shape)


**Vad output visar:** bekräftar att testsetet (samma `data/test.csv` som skapades i
notebook 01 och aldrig använts för träning) samt den sparade modellen från notebook 04
laddats in korrekt.


In [ ]:
y_pred_probs = sentiment_model.predict([X_test_ids, X_test_mask])
y_pred = np.argmax(y_pred_probs, axis=1)

sentiment_accuracy = accuracy_score(y_test, y_pred)
sentiment_f1 = f1_score(y_test, y_pred, average="macro")

print("Accuracy:", round(sentiment_accuracy, 3))
print("F1 (macro):", round(sentiment_f1, 3))
print()
print(classification_report(y_test, y_pred, target_names=[id2label[i] for i in range(3)]))


**Vad output visar:** **Accuracy** är andelen tweets modellen klassificerade rätt av
totalt 540 i testsetet. **F1 (macro)** väger ihop precision och recall och räknar varje klass
lika mycket — viktigt eftersom klasserna är ojämnt fördelade (63% negative). Om macro-F1 är
klart lägre än accuracy betyder det att modellen är sämre på minoritetsklasserna (troligen
**positive**, som har färst exempel).

`classification_report` bryter ner **precision** (av alla tweets modellen kallade t.ex.
"negative", hur stor andel var faktiskt negativa?), **recall** (av alla faktiskt negativa
tweets, hur stor andel hittade modellen?) och **f1-score** per klass, plus `support`
(antal riktiga exempel av den klassen i testsetet).


In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[id2label[i] for i in range(3)],
    yticklabels=[id2label[i] for i in range(3)],
    ax=ax,
)
ax.set_xlabel("Predikterat"); ax.set_ylabel("Verkligt")
ax.set_title("Confusion matrix — sentiment (test set)")
plt.tight_layout()
plt.savefig("charts/sentiment_confusion_matrix.png", dpi=150)
plt.show()


**Vad diagrammet visar:** varje rad är den **verkliga** klassen, varje kolumn är vad
modellen **predikterade**. Diagonalen (uppe-vänster till nere-höger) är rätta svar — ju
mörkare/högre siffror där, desto bättre. Titta särskilt på vilka celler **utanför**
diagonalen som har höga värden: om t.ex. många verkliga "neutral"-tweets hipnas som
"negative" säger det något om var modellens gränsdragning är som otydligast.


## Del B — Utvärdera orsaksmodellen (notebook 05)

In [ ]:
reason_test_df = pd.read_csv("data/reason_test.csv")
reason_model = tf.keras.models.load_model("models/reason_model.keras")
reason_tokenizer = DistilBertTokenizerFast.from_pretrained("models/reason_tokenizer")

with open("models/reason_classes.json") as f:
    reason_classes = json.load(f)
reason2id = {r: i for i, r in enumerate(reason_classes)}
id2reason = {i: r for r, i in reason2id.items()}

X_rtest_ids, X_rtest_mask = tokenize(reason_test_df["text"], reason_tokenizer)
y_rtest = reason_test_df["reason_group"].map(reason2id).values

print("Testset (orsak):", reason_test_df.shape, " Klasser:", reason_classes)


**Vad output visar:** bekräftar att orsaksmodellens eget testset (`reason_test.csv`,
skapat i notebook 05 — inte samma testset som sentimentmodellen!) och de sparade
klassnamnen laddats korrekt.


In [ ]:
y_rpred_probs = reason_model.predict([X_rtest_ids, X_rtest_mask])
y_rpred = np.argmax(y_rpred_probs, axis=1)

reason_accuracy = accuracy_score(y_rtest, y_rpred)
reason_f1 = f1_score(y_rtest, y_rpred, average="macro")

print("Accuracy:", round(reason_accuracy, 3))
print("F1 (macro):", round(reason_f1, 3))
print()
print(classification_report(y_rtest, y_rpred, target_names=reason_classes))


**Vad output visar:** samma typ av mått som för sentimentmodellen, fast för 6
klasser istället för 3. Förvänta lägre accuracy än sentimentmodellen — fler klasser gör
uppgiften statistiskt svårare, och "Other" är per definition en spännvidd av olika sorters
klagomål vilket gör den klassen svårare att träffa rätt på.


In [ ]:
cm_reason = confusion_matrix(y_rtest, y_rpred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_reason, annot=True, fmt="d", cmap="Oranges",
    xticklabels=reason_classes, yticklabels=reason_classes, ax=ax,
)
ax.set_xlabel("Predikterat"); ax.set_ylabel("Verkligt")
ax.set_title("Confusion matrix — orsak (test set)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig("charts/reason_confusion_matrix.png", dpi=150)
plt.show()


**Vad diagrammet visar:** samma princip som för sentiment-matrisen. Håll särskilt
utkik efter förväxlingar med **Other** — eftersom den klassen är en blandning av många olika
ursprungliga kategorier är det naturligt att den både "läcker" tweets till och får tweets
från de andra, mer specifika klasserna.


## Del C — Koppla ihop med den oförvanskade klustringen (notebook 03)

I notebook 03 såg vi att råa, oförtränade DistilBERT-embeddings gav en **svag**
koppling till sentiment (Adjusted Rand Index någonstans runt 0.1–0.4). Jämför det talet med
accuracy-siffran från Del A här ovanför — skillnaden är ett konkret mått på **värdet av
supervised fine-tuning**: hur mycket bättre modellen blir när den faktiskt får lära sig av
etiketterad data, jämfört med att bara använda generell språkförståelse.


## Sammanfattning av alla resultat

In [ ]:
summary = pd.DataFrame({
    "Modell": ["Sentiment (04)", "Orsak (05)"],
    "Antal klasser": [3, len(reason_classes)],
    "Accuracy": [round(sentiment_accuracy, 3), round(reason_accuracy, 3)],
    "F1 (macro)": [round(sentiment_f1, 3), round(reason_f1, 3)],
})
summary


**Vad tabellen visar:** en sammanfattning sida vid sida av båda modellerna — bra
att klistra in direkt i presentationens resultatslide. Förvänta att sentimentmodellen (3
klasser) presterar något bättre än orsaksmodellen (6 klasser), vilket är naturligt givet
antalet klasser och är inget tecken på att något är fel.


## Slutsatser & nästa steg

- **Steg 6 (utvärdering)** är nu klart för båda modellerna, med accuracy, F1, precision/recall
  per klass och confusion matrices.
- Om resultaten inte är tillräckligt bra: prova fler fine-tuning-epoker, lås upp ytterligare
  ett transformerblock, eller träna på hela det ursprungliga datasetet (14 640 tweets)
  istället för det nedsamplade urvalet.
- **Steg 7 (implementering)** finns redan färdig som en Streamlit-app — koppla in båda
  modellerna där för en fullständig kundtjänst-triage: *vad* kunden känner OCH *varför*.

Det här avslutar hela pipelinen: `01` → `02` → `03` → `04` → `05` → `06`.
